In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql.functions import *


# creation spark session
spark = SparkSession.builder.appName("Assignment_2").enableHiveSupport().getOrCreate()

#hdfs paths
movies_hdfs_path = '/assignment_2_data/movies.csv'
ratings_hdfs_path = '/assignment_2_data/ratings.csv'
tags_hdfs_path = '/assignment_2_data/tags.csv'

26/10/23 08:49:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# creating movies dataframe
movies_schema = '''
                    movieId INT,
                    title STRING,
                    genres STRING
                ''' 
movies_df = spark.read.format('csv').schema(movies_schema).option('header',True).load(movies_hdfs_path)

movies_df = movies_df.withColumn("movie_name",regexp_extract(col("title"), r"^(.*?)\s*\(\d{4}\)$", 1))\
.withColumn("year",regexp_extract(col("title"), r"\((\d{4})\)$", 1).cast("int"))\
.withColumn("genres",split(col("genres"),r"\|"))
movies_df = movies_df.select("movieId","movie_name","year","genres")
movies_df.show()
movies_df.printSchema()

+-------+--------------------+----+--------------------+
|movieId|          movie_name|year|              genres|
+-------+--------------------+----+--------------------+
|      1|           Toy Story|1995|[Adventure, Anima...|
|      2|             Jumanji|1995|[Adventure, Child...|
|      3|    Grumpier Old Men|1995|   [Comedy, Romance]|
|      4|   Waiting to Exhale|1995|[Comedy, Drama, R...|
|      5|Father of the Bri...|1995|            [Comedy]|
|      6|                Heat|1995|[Action, Crime, T...|
|      7|             Sabrina|1995|   [Comedy, Romance]|
|      8|        Tom and Huck|1995|[Adventure, Child...|
|      9|        Sudden Death|1995|            [Action]|
|     10|           GoldenEye|1995|[Action, Adventur...|
|     11|American Presiden...|1995|[Comedy, Drama, R...|
|     12|Dracula: Dead and...|1995|    [Comedy, Horror]|
|     13|               Balto|1995|[Adventure, Anima...|
|     14|               Nixon|1995|             [Drama]|
|     15|    Cutthroat Island|1

In [3]:
# creating ratings dataframe

ratings_schema = '''
userId INT,
movieID INT,
rating DOUBLE,
timestamp INT
'''
ratings_df = spark.read.format("csv").schema(ratings_schema).option("header",True).load(ratings_hdfs_path)

ratings_df = ratings_df.withColumn("date",to_date(from_unixtime(col("timestamp"))))\
.withColumn("year", year(col("date")))\
.withColumn("month", date_format(col("date"), "MMMM"))

ratings_df.show()

+------+-------+------+---------+----------+----+-----+
|userId|movieID|rating|timestamp|      date|year|month|
+------+-------+------+---------+----------+----+-----+
|     1|      1|   4.0|964982703|2000-07-30|2000| July|
|     1|      3|   4.0|964981247|2000-07-30|2000| July|
|     1|      6|   4.0|964982224|2000-07-30|2000| July|
|     1|     47|   5.0|964983815|2000-07-30|2000| July|
|     1|     50|   5.0|964982931|2000-07-30|2000| July|
|     1|     70|   3.0|964982400|2000-07-30|2000| July|
|     1|    101|   5.0|964980868|2000-07-30|2000| July|
|     1|    110|   4.0|964982176|2000-07-30|2000| July|
|     1|    151|   5.0|964984041|2000-07-30|2000| July|
|     1|    157|   5.0|964984100|2000-07-30|2000| July|
|     1|    163|   5.0|964983650|2000-07-30|2000| July|
|     1|    216|   5.0|964981208|2000-07-30|2000| July|
|     1|    223|   3.0|964980985|2000-07-30|2000| July|
|     1|    231|   5.0|964981179|2000-07-30|2000| July|
|     1|    235|   4.0|964980908|2000-07-30|2000

In [4]:
# Question 1 - Show the aggregated number of ratings per year
num_of_ratings_per_year_df = ratings_df.groupBy("year").count().withColumnRenamed("count","number_of_ratings").orderBy("year")

num_of_ratings_per_year_df.show()

# num_of_ratings_per_year_df.write.format('csv').mode('overwrite')\
# .option('header', 'true').option('delimiter', ',')\
# .save('/assignment_2_data/outputs/question_1')

# print("Write Successful!!!")

+----+-----------------+
|year|number_of_ratings|
+----+-----------------+
|1996|             6040|
|1997|             1916|
|1998|              507|
|1999|             2439|
|2000|            10061|
|2001|             3922|
|2002|             3478|
|2003|             4014|
|2004|             3279|
|2005|             5813|
|2006|             4059|
|2007|             7114|
|2008|             4351|
|2009|             4158|
|2010|             2301|
|2011|             1690|
|2012|             4656|
|2013|             1664|
|2014|             1439|
|2015|             6616|
+----+-----------------+
only showing top 20 rows



In [5]:
# Question 2 - Show the average monthly number of ratings

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")

ratings_df.createTempView("ratings_view")

sql_query = """SELECT year, month, avg(rating) as avg_monthly_num_of_ratings 
FROM ratings_view 
GROUP BY year,month
ORDER BY year desc ,month desc
LIMIT 10"""

df_avg_monthly_num_ratings = spark.sql(sql_query)

# df_avg_monthly_num_ratings.write.format('csv').mode('overwrite')\
# .option('header', 'true').option('delimiter', ',')\
# .save('/assignment_2_data/outputs/question_2')

# print("Write Successful!!!")

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


In [6]:
# Question 3 - Show ratings level distribution

from pyspark.sql.window import Window

# we can use this sql query to get answer of this question, but I will write PySpark code for this now
sql_query = """
WITH CTE_ratings AS (
    SELECT rating,
    CASE 
        WHEN rating BETWEEN 0 and 2 THEN '0.0 - 2.0'
        WHEN rating BETWEEN 2.1 and 4 THEN '2.1 - 4.0'
        ELSE '> 4'
    END AS ratings_bucket
    FROM ratings_view
),

CTE_ratings_aggregations AS (
    SELECT ratings_bucket, 
    COUNT(*) as counts
    FROM CTE_ratings
    GROUP BY ratings_bucket
)

SELECT 
    ratings_bucket,
    counts,
    (counts * 100.0 / SUM(counts) OVER ()) AS percentage
FROM CTE_ratings_aggregations
ORDER BY ratings_bucket
"""

ratings_bucket_df = ratings_df.withColumn(
    "ratings_bucket",
    when((col("rating") > 0.0) & (col("rating") <= 2.0), "0.0 - 2.0")
    .when((col("rating") > 2.0) & (col("rating") <= 4.0), "2.0 - 4.0")
    .otherwise("> 4.0")
)

ratings_bucket_df = ratings_bucket_df.select("rating","ratings_bucket")

ratings_aggregations_df = ratings_bucket_df.groupBy("ratings_bucket").agg(count("*").alias("counts"))

ratings_distribution_df = ratings_aggregations_df.withColumn("percentage",
                          round(col("counts")*100.0/sum("counts").over(Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)),2)).\
                        orderBy("ratings_bucket")

ratings_distribution_df.show()

# ratings_distribution_df.write.format('csv').mode('overwrite')\
# .option('header', 'true').option('delimiter', ',')\
# .save('/assignment_2_data/outputs/question_3')

# print("Write Successful!!!")


26/10/23 08:49:32 INFO PlanChangeLogger: 
 Dataproc Rule org.apache.spark.sql.catalyst.optimizer.google.EncodeStringCaseWhenInAggregate effective 1 times.

26/10/23 08:49:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:49:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:49:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 08:49:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:49:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 08:49:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:49:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------------+------+----------+
|ratings_bucket|counts|percentage|
+--------------+------+----------+
|     0.0 - 2.0| 13523|     13.41|
|     2.0 - 4.0| 65551|     65.01|
|         > 4.0| 21762|     21.58|
+--------------+------+----------+



In [7]:
# creating tags dataframe

tags_schema = '''
                    userId INT,
                    movieID STRING,
                    tag STRING,
                    timestamp INT
                ''' 
tags_df = spark.read.format('csv').schema(tags_schema).option('header',True).load(tags_hdfs_path)

In [8]:
# Question 4 - Show the 18 movies that are tagged but not rated

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")


if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

sql_query = """
    WITH CTE_movie_ids AS (
    SELECT DISTINCT tv.movieID
    FROM tags_view as tv
    LEFT JOIN ratings_view as rt
    ON tv.movieID = rt.movieID
    WHERE rt.movieID is NULL)
    
    SELECT mv.movie_name
    FROM movies_view as mv
    JOIN CTE_movie_ids as id
    ON id.movieID = mv.movieID
"""

# pyspark code for the same:
movie_ids_df_q4 = tags_df.join(ratings_df,on="movieID",how="left_anti").select("movieId").distinct()
movie_name_df_q4 = movie_ids_df_q4.join(movies_df,on="movieID",how="inner").select("movie_name")

tagged_not_rated_movies_df = spark.sql(sql_query)
tagged_not_rated_movies_df.show()

# tagged_not_rated_movies_df.write.format('csv').mode('overwrite')\
# .option('header', 'true').option('delimiter', ',')\
# .save('/assignment_2_data/outputs/question_4')

# print("Write Successful!!!")

26/10/23 08:49:35 INFO PlanChangeLogger: 
 Dataproc Rule org.apache.spark.sql.catalyst.optimizer.google.ConvertToAntiJoin effective 1 times.



+--------------------+
|          movie_name|
+--------------------+
|      Innocents, The|
|             Niagara|
|     For All Mankind|
|Color of Paradise...|
|I Know Where I'm ...|
|         Chosen, The|
|Road Home, The (W...|
|             Scrooge|
|               Proof|
|  Parallax View, The|
|   This Gun for Hire|
|Roaring Twenties,...|
|Mutiny on the Bounty|
|In the Realms of ...|
|   Twentieth Century|
|  Call Northside 777|
|Browning Version,...|
|         Chalet Girl|
+--------------------+



In [17]:
# Question 5 - Show movies that have ratings but no tag

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

sql_query = """
    WITH CTE_movie_ids AS (
    SELECT DISTINCT rt.movieID
    FROM ratings_view as rt 
    LEFT JOIN tags_view as tv
    ON rt.movieID = tv.movieID
    WHERE tv.movieID is NULL)
    
    SELECT mv.movie_name
    FROM movies_view as mv
    JOIN CTE_movie_ids as id
    ON id.movieID = mv.movieID
"""

# pyspark code for the same:
movie_ids_df_q5 = ratings_df.join(tags_df,on="movieID",how="left_anti").select("movieId").distinct()
movie_name_df_q5 = movie_ids_df_q5.join(movies_df,on="movieID",how="inner").select("movie_name")

rated_untagged_movies_df = spark.sql(sql_query)
rated_untagged_movies_df.show()

# rated_untagged_movies_df.write.format('csv').mode('overwrite')\
# .option('header', 'true').option('delimiter', ',')\
# .save('/assignment_2_data/outputs/question_5')

# print("Write Successful!!!")


+--------------------+
|          movie_name|
+--------------------+
|           King Kong|
|            Candyman|
|               Spawn|
|Land Before Time,...|
|    High School High|
|Hellbound: Hellra...|
|  The Electric Hotel|
|Awfully Big Adven...|
|What Happened Was...|
|         Moonwalkers|
|        High Society|
|            Sky High|
|               Gordy|
|    Running Man, The|
|Fearless Vampire ...|
|             Tin Men|
|        Wolfman, The|
|       Comme un chef|
|    House Bunny, The|
|       Ice Cream Man|
+--------------------+
only showing top 20 rows



In [18]:
# Question 6 - Focusing on the rated untagged movies with more than 30 user ratings, show the top 10 movies in terms of average rating and number of ratings

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

sql_query = """
    WITH CTE_total_avg_ratings AS
    (
    SELECT rt.movieID,
    COUNT(*) as total_ratings,
    AVG(rt.rating) as avg_rating
    FROM ratings_view as rt 
    LEFT JOIN tags_view as tv
    ON rt.movieID = tv.movieID
    WHERE tv.movieID is NULL
    GROUP BY rt.movieID
    HAVING COUNT(*) > 30
    ),
    
    CTE_movie_names_total_avg_ratings AS
    (
    SELECT mv.movie_name, tar.total_ratings, tar.avg_rating
    FROM CTE_total_avg_ratings as tar
    JOIN movies_view as mv
    ON tar.movieID = mv.movieID
    ),
    
    CTE_count_rank AS
    (
    SELECT 
        movie_name,
        total_ratings,
        DENSE_RANK() OVER(ORDER BY total_ratings DESC) as count_rank
    FROM CTE_movie_names_total_avg_ratings
    ),
    
    CTE_avg_rank AS
    (
    SELECT 
        movie_name,
        avg_rating,
        DENSE_RANK() OVER(ORDER BY avg_rating DESC) as avg_rank
    FROM CTE_movie_names_total_avg_ratings
    )
    
    SELECT cr.movie_name, cr.total_ratings, cr.count_rank, ar.movie_name, ar.avg_rating, ar.avg_rank
    FROM CTE_count_rank cr
    JOIN CTE_avg_rank ar
    ON cr.count_rank = ar.avg_rank
    WHERE cr.count_rank <=10 AND ar.avg_rank <= 10
"""

top_rated_untagged_movies_df = spark.sql(sql_query)
top_rated_untagged_movies_df.show()

# top_rated_untagged_movies_df.write.format('csv').mode('overwrite')\
# .option('header', 'true').option('delimiter', ',')\
# .save('/assignment_2_data/outputs/question_6')

# print("Write Successful!!!")

# ratings_df.show()
# tags_df.show()
# movies_df.show()

26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 0

26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 08:52:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 08:52:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------------------+-------------+----------+--------------------+-----------------+--------+
|          movie_name|total_ratings|count_rank|          movie_name|       avg_rating|avg_rank|
+--------------------+-------------+----------+--------------------+-----------------+--------+
|     American Beauty|          204|         1|Boondock Saints, The| 4.22093023255814|       1|
|Ace Ventura: Pet ...|          161|         2|              Brazil|4.177966101694915|       2|
|           Mask, The|          157|         3|Cinema Paradiso (...|4.161764705882353|       3|
|            Die Hard|          145|         4|              Snatch|4.155913978494624|       4|
|Die Hard: With a ...|          144|         5|For a Few Dollars...|4.151515151515151|       5|
|       Groundhog Day|          143|         6|Lives of Others, ...|4.117647058823529|       6|
|Dumb & Dumber (Du...|          133|         7|         Toy Story 3|4.109090909090909|       7|
|           GoldenEye|          132|    

In [19]:
# Question 7 - What is the average number of tags per movie in tagsDF? And the average number of tags per user? How does it compare with the average number of 
# tags a user assigns to a movie?

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

# Avg number of tags per movie in tags df

avg_query = """
WITH CTE_avg_tags_per_movie AS (
    SELECT ROUND(SUM(total_count)/COUNT(movieID),2) as avg_tags_per_movie FROM 
    (
        SELECT movieID, COUNT(*) as total_count from tags_view GROUP BY movieID ORDER BY movieID 
    )
),

CTE_avg_tags_per_user AS (
    SELECT ROUND(SUM(total_count)/COUNT(userID),2) as avg_tags_per_user FROM 
    (
        SELECT userID, COUNT(*) as total_count from tags_view GROUP BY userID ORDER BY userID 
    )
)

SELECT 
    atm.avg_tags_per_movie, atu.avg_tags_per_user,
    CASE
        WHEN atm.avg_tags_per_movie > atu.avg_tags_per_user THEN "tags per movie are greater than tags per user"
        WHEN atm.avg_tags_per_movie < atu.avg_tags_per_user THEN "tags per user are greater than tags per movie"
        ELSE "both metrics are same"
    END AS Comparison
FROM CTE_avg_tags_per_movie as atm, CTE_avg_tags_per_user as atu
"""

avg_tags_per_movie_df = tags_df.groupBy('movieID').agg(count('tag').alias('total_count'))
avg_tags_per_movie_df = avg_tags_per_movie_df.agg(round(sum('total_count')/count('movieID'),2).alias('avg_tags_per_movie'))

avg_tags_per_user_df = tags_df.groupBy('userID').agg(count('tag').alias('total_count'))
avg_tags_per_user_df = avg_tags_per_user_df.agg(round(sum('total_count')/count('userID'),2).alias('avg_tags_per_user'))

comparision_df = avg_tags_per_movie_df.crossJoin(avg_tags_per_user_df)
comparision_df = comparision_df.withColumn("comparision",
                                          when(col("avg_tags_per_movie")>col("avg_tags_per_user"), "Tags per movie are greater than tags per user").
                                          when(col("avg_tags_per_movie")<col("avg_tags_per_user"), "Tags per user are greater than tags per movie").
                                          otherwise("Both metrics are the same"))

comparision_df.show(truncate = False)

comparision_df.write.format('csv').mode('overwrite')\
.option('header', 'true').option('delimiter', ',')\
.save('/assignment_2_data/outputs/question_7')

print("Write Successful!!!")


+------------------+-----------------+---------------------------------------------+
|avg_tags_per_movie|avg_tags_per_user|comparision                                  |
+------------------+-----------------+---------------------------------------------+
|2.34              |63.5             |Tags per user are greater than tags per movie|
+------------------+-----------------+---------------------------------------------+



Write Successful!!!


In [20]:
# Question 8 - Identify the users that tagged movies without rating them

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

movies_without_rating_query = """
SELECT distinct tv.userID
FROM tags_view tv
LEFT JOIN ratings_view rv
ON tv.movieID = rv.movieID
WHERE rv.userID is NULL
"""

movies_without_rating_df = tags_df.alias('td').join(ratings_df.alias('rd'),on="movieID",how="left").\
filter(col("rd.userID").isNull()).select(col("td.userID")).alias("userID").distinct()
movies_without_rating_df.show()

movies_without_rating_df.write.format('csv').mode('overwrite')\
.option('header', 'true').option('delimiter', ',')\
.save('/assignment_2_data/outputs/question_8')

print("Write Successful!!!")


+------+
|userID|
+------+
|   474|
|   288|
|   543|
|   318|
+------+



Write Successful!!!


In [21]:
# Question 9 - What is the average number of ratings per user in ratings DF? And the average number of ratings per movie?

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

avg_rating_per_user_query = """
WITH CTE_avg_rating_per_user AS
(
    SELECT ROUND(SUM(total_ratings_count)/COUNT(userID),2) as avg_rating_per_user FROM
    (
        SELECT userID, count(*) as total_ratings_count FROM ratings_view GROUP BY userID
    )
),

CTE_avg_rating_per_movie AS
(
    SELECT ROUND(SUM(total_ratings_count)/COUNT(movieID),2) as avg_rating_per_movie FROM
    (
        SELECT movieID, count(*) as total_ratings_count FROM ratings_view GROUP BY movieID
    )
)

SELECT aru.avg_rating_per_user, arm.avg_rating_per_movie
FROM CTE_avg_rating_per_user as aru, CTE_avg_rating_per_movie as arm

"""

avg_rating_per_user_df = ratings_df.groupBy('userID').agg(count('rating').alias('total_ratings_count'))
avg_rating_per_user_df = avg_rating_per_user_df.agg(round(sum('total_ratings_count')/count('userID'),2).alias('avg_rating_per_user'))

avg_rating_per_movie_df = ratings_df.groupBy('movieID').agg(count('rating').alias('total_ratings_count'))
avg_rating_per_movie_df = avg_rating_per_movie_df.agg(round(sum('total_ratings_count')/count('movieID'),2).alias('avg_rating_per_movie'))

combined_df = avg_rating_per_user_df.crossJoin(avg_rating_per_movie_df)

combined_df.write.format('csv').mode('overwrite')\
.option('header', 'true').option('delimiter', ',')\
.save('/assignment_2_data/outputs/question_9')

print("Write Successful!!!")

Write Successful!!!


In [22]:
# Question 10 - What is the predominant (frequency based) genre per rating level?

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

exploded_movies_df = movies_df.withColumn('genre',explode(col("genres"))).drop("genres")
if spark.catalog.tableExists("exploded_movies_view"):
    spark.catalog.dropTempView("exploded_movies_view")
exploded_movies_df.createTempView("exploded_movies_view")

query = """
WITH CTE_rating_genre_count AS
(
    SELECT rating, genre, COUNT(genre) as total_count
    FROM (
        SELECT *
        FROM exploded_movies_view emv
        JOIN ratings_view rv
        ON emv.movieID = rv.movieID
    )
    GROUP BY rating, genre
    ORDER BY rating DESC, total_count DESC
),

CTE_rank as
(
    SELECT rating, genre, total_count,
    DENSE_RANK() OVER(PARTITION BY rating ORDER BY total_count DESC) as frequency
    FROM CTE_rating_genre_count
)

SELECT rating, genre
FROM CTE_rank
WHERE frequency = 1
ORDER BY rating DESC
"""

genre_frequency_per_rating_df = spark.sql(query)

genre_frequency_per_rating_df.write.format('csv').mode('overwrite')\
.option('header', 'true').option('delimiter', ',')\
.save('/assignment_2_data/outputs/question_10')

print("Write Successful!!!")

Write Successful!!!


In [23]:
# Question 11.1 - What is the predominant tag per genre and the most tagged genres?

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

exploded_movies_df = movies_df.withColumn('genre',explode(col("genres"))).drop("genres")
if spark.catalog.tableExists("exploded_movies_view"):
    spark.catalog.dropTempView("exploded_movies_view")
exploded_movies_df.createTempView("exploded_movies_view")

query = """
WITH CTE_genre_tag_count AS
(
    SELECT genre, tag, count(tag) as total_count
    FROM
    (
        SELECT *
        FROM exploded_movies_view emv
        JOIN tags_view tv
        ON emv.movieID = tv.movieID
    )
    GROUP BY genre, tag
),
CTE_rank as
(
    SELECT genre, tag, total_count,
    DENSE_RANK() OVER(PARTITION BY genre ORDER BY total_count DESC) as frequency
    FROM CTE_genre_tag_count
)

SELECT genre, tag
FROM CTE_rank
WHERE frequency = 1
"""

genre_tag_count_df = spark.sql(query)


genre_tag_count_df.write.format('csv').mode('overwrite')\
.option('header', 'true').option('delimiter', ',')\
.save('/assignment_2_data/outputs/question_11_1')

print("Write Successful!!!")


Write Successful!!!


In [25]:
# Question 11.2 - What is the predominant tag per genre and the most tagged genres?

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

exploded_movies_df = movies_df.withColumn('genre',explode(col("genres"))).drop("genres")
if spark.catalog.tableExists("exploded_movies_view"):
    spark.catalog.dropTempView("exploded_movies_view")
exploded_movies_df.createTempView("exploded_movies_view")

query = '''
WITH CTE_most_tagged_genre AS
(
 SELECT genre, count(tag) as total_count
    FROM
    (
        SELECT *
        FROM exploded_movies_view emv
        JOIN tags_view tv
        ON emv.movieID = tv.movieID
    )
    GROUP BY genre
    ORDER BY total_count DESC
),

CTE_rank as
(
    SELECT 
        genre,
        DENSE_RANK() OVER(ORDER BY total_count DESC) as genre_rank
    FROM CTE_most_tagged_genre
)

SELECT genre FROM CTE_rank where genre_rank = 1
'''

most_tagged_genre = spark.sql(query)

most_tagged_genre.write.format('csv').mode('overwrite')\
.option('header', 'true').option('delimiter', ',')\
.save('/assignment_2_data/outputs/question_11_2')

print("Write Successful!!!")


26/10/23 09:42:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 09:42:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 09:42:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 09:42:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 09:42:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 09:42:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 09:42:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 09:42:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 09:42:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 09:42:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 09:42:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Write Successful!!!


In [35]:
# Question 12 - What are the most predominant (popularity based) movies?

if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

sql_query = """
WITH CTE_ratings_count AS
(
    SELECT 
        mv.movie_name, 
        count(rv.rating) num_of_ratings
    FROM ratings_view as rv
    LEFT JOIN movies_view as mv
    ON rv.movieID = mv.movieID
    GROUP BY mv.movie_name
),

CTE_rank AS
(
    SELECT 
        movie_name, 
        num_of_ratings,
        DENSE_RANK() OVER(ORDER BY num_of_ratings DESC) as movie_rank
    FROM CTE_ratings_count
)

SELECT * FROM CTE_rank WHERE movie_rank <= 10
"""
most_popular_movies_df = spark.sql(sql_query)

most_popular_movies_df.write.format('csv').mode('overwrite')\
.option('header', 'true').option('delimiter', ',')\
.save('/assignment_2_data/outputs/question_12')

print("Write Successful!!!")

26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:05:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Write Successful!!!


In [40]:
# Question 12 - Top 10 movies in terms of average rating (provided more than 30 users reviewed them)
if spark.catalog.tableExists("ratings_view"):
    spark.catalog.dropTempView("ratings_view")
ratings_df.createTempView("ratings_view")

if spark.catalog.tableExists("movies_view"):
    spark.catalog.dropTempView("movies_view")
movies_df.createTempView("movies_view")

if spark.catalog.tableExists("tags_view"):
    spark.catalog.dropTempView("tags_view")
tags_df.createTempView("tags_view")

sql_query = """
WITH CTE_ratings_count AS
(
    SELECT 
        mv.movie_name, 
        count(rv.rating) num_of_ratings,
        AVG(rv.rating) avg_rating
    FROM ratings_view as rv
    LEFT JOIN movies_view as mv
    ON rv.movieID = mv.movieID
    GROUP BY mv.movie_name
),

CTE_rank AS
(
    SELECT 
        movie_name, 
        avg_rating,
        DENSE_RANK() OVER(ORDER BY avg_rating DESC) as movie_rank
    FROM CTE_ratings_count
    WHERE num_of_ratings > 30
    
)

SELECT * FROM CTE_rank WHERE movie_rank <= 10
"""
most_popular_movies_in_ratings_df = spark.sql(sql_query)

most_popular_movies_in_ratings_df.write.format('csv').mode('overwrite')\
.option('header', 'true').option('delimiter', ',')\
.save('/assignment_2_data/outputs/question_13')

print("Write Successful!!!")

26/10/23 10:20:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:20:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:20:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 10:20:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:20:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 10:20:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:20:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/23 10:20:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/23 10:20:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Write Successful!!!
